# GP-TEMPEST Training Notebook
Run this on Kaggle with a GPU enabled (Settings → Accelerator → GPU T4 x2).

In [ ]:
# Install GP-TEMPEST directly from GitHub
!pip install git+https://github.com/moldyn/GP-TEMPEST.git -q

In [ ]:
import numpy as np
import torch
from gptempest import TEMPEST, MaternKernel
from gptempest.utils import load_prepare_data

print(f'PyTorch version: {torch.__version__}')
print(f'GPU available:   {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:             {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
# Paths (adjust to your Kaggle dataset mount point)
DATA_PATH            = '/kaggle/input/your-dataset/data.dat'
INDUCING_POINTS_PATH = '/kaggle/input/your-dataset/inducing_points.dat'
SAVE_PATH            = '/kaggle/working/'

# Architecture
DIM_INPUT   = 2        # number of input features
DIM_LATENT  = 2        # latent space dimensionality
LAYERS      = [32, 32, 32]  # hidden layer sizes (decoder = reversed)

# Training
EPOCHS        = 100
BATCH_SIZE    = 1024
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-6

# GP kernel
BETA         = 50.0   # GP loss weight
KERNEL_NU    = 1.5    # Matern smoothness: 0.5, 1.5, or 2.5
KERNEL_SCALE = 1e3    # time scale

CUDA  = torch.cuda.is_available()
DTYPE = torch.float64
torch.manual_seed(42)
if CUDA:
    torch.cuda.manual_seed(42)

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
dataset         = load_prepare_data(DATA_PATH, DTYPE)
inducing_points = np.loadtxt(INDUCING_POINTS_PATH)
print(f'Dataset size:          {len(dataset)} frames')
print(f'Inducing points shape: {inducing_points.shape}')

In [ ]:
# ── Build model ───────────────────────────────────────────────────────────────
kernel = MaternKernel(nu=KERNEL_NU, scale=KERNEL_SCALE, dtype=DTYPE)

model = TEMPEST(
    cuda=CUDA,
    kernel=kernel,
    dim_input=DIM_INPUT,
    dim_latent=DIM_LATENT,
    layers_hidden_encoder=LAYERS,
    layers_hidden_decoder=LAYERS[::-1],
    inducing_points=inducing_points,
    beta=BETA,
    N_data=len(dataset),
    dtype=DTYPE,
)
print(model)

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
model.train_model(
    dataset,
    train_size=1,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
)

model_path = f'{SAVE_PATH}/model.pt'
torch.save(model.state_dict(), model_path)
print(f'Model saved to {model_path}')

In [ ]:
# ── Extract latent space ──────────────────────────────────────────────────────
embedding = model.extract_latent_space(dataset, BATCH_SIZE)
embedding_path = f'{SAVE_PATH}/embedding.dat'
np.savetxt(embedding_path, embedding, fmt='%.4f')
print(f'Embedding shape: {embedding.shape}')
print(f'Saved to {embedding_path}')

In [ ]:
# ── Quick plot ────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(embedding[:, 0], embedding[:, 1], s=1, alpha=0.5)
ax.set_xlabel('$z_1$')
ax.set_ylabel('$z_2$')
ax.set_title('Latent space')
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}/embedding.png', dpi=150)
plt.show()